<p style="text-align:center">
    <img src="https://media.licdn.com/dms/image/v2/D560BAQGRKQWaOldXvg/company-logo_200_200/company-logo_200_200/0/1723551442229/cambo_ai_logo?e=2147483647&v=beta&t=FrK5k676UJMjoOULl4R3euChg1VvClew28fxNscSd9E" width="80" alt="CamboAI Logo">
</p>

# **I. Objectives**

**Estimated time needed: 30 minutes**

**After completing this lab you will be able to:**
- Know how to use sql in MySQL & Postgresql Databases.
- Perform **CRUD** Operation in SQL.
- Perform ETL From **Source** (MySQL) to **Destination** (PostgreSQL) Database.

# **II. Import Libraries**

In [17]:
# install pymysql, psycopg2 libraries
%pip install pymysql psycopg2 --quiet

Note: you may need to restart the kernel to use updated packages.


In [18]:
# import libraries
import pymysql
import psycopg2
import pandas as pd

# filter warning
import warnings
warnings.filterwarnings('ignore')

# **III. Extract the Data from Source**

In [19]:
# step 1: extract only new or updated records from MySQL
def extract_incremental_from_mysql(mysql_config, query, last_loaded_value):
    # Connect to MySQL database
    connection = pymysql.connect(
        host=mysql_config['host'],
        user=mysql_config['user'],
        password=mysql_config['password'],
        database=mysql_config['database']
    )

    # modify the query to include incremental condition
    query = query.format(last_loaded_value)

    # execute query and fetch data into a DataFrame
    df = pd.read_sql(query, connection)

    # close MySQL connection
    connection.close()

    return df

# **IV. Transform the Data**

In [20]:
# step 2: transform the data (optional)
def transform_data(df):
    # example transformation: renaming columns to lowercase
    df.columns = [col.lower() for col in df.columns]

    # other transformations can be done here
    return df

# **V. Load the Data Into Target**

In [21]:
# step 3: load data into PostgreSQL
def load_to_postgres(postgres_config, df, table_name):
    # connect to PostgreSQL database
    connection = psycopg2.connect(
        host=postgres_config['host'],
        user=postgres_config['user'],
        password=postgres_config['password'],
        database=postgres_config['database']
    )

    cursor = connection.cursor()

    # create table if it doesn't exist (example schema, modify as needed)
    create_table_query = f'''
    CREATE TABLE IF NOT EXISTS {table_name} (
        id SERIAL PRIMARY KEY,
        {", ".join([f"{col} TEXT" for col in df.columns if col != 'id'])}  -- Exclude id from column creation
    )
    '''
    cursor.execute(create_table_query)
    connection.commit()

    # upsert data into PostgreSQL
    for i, row in df.iterrows():
        # define the insert query with upsert logic
        insert_query = f'''
        INSERT INTO {table_name} ({", ".join(df.columns)})
        VALUES ({", ".join(["%s"] * len(df.columns))})
        ON CONFLICT (id) DO UPDATE SET
            {", ".join([f"{col} = EXCLUDED.{col}" for col in df.columns if col != 'id'])};  -- Update all columns except `id`
        '''

        # execute the insert query with the row values
        cursor.execute(insert_query, tuple(row))

    connection.commit()

    # close PostgreSQL connection
    cursor.close()
    connection.close()

# **VI. Fetch the Last Loaded Timestamp from Target Table**

In [22]:
# step 4: fetch the last loaded timestamp from PostgreSQL
def get_last_loaded_value(postgres_config, table_name, updated_at):
    connection = psycopg2.connect(
        host=postgres_config['host'],
        user=postgres_config['user'],
        password=postgres_config['password'],
        database=postgres_config['database']
    )
    cursor = connection.cursor()

    # get the maximum value of the identifier column timestamp
    query = f'SELECT MAX(updated_at) FROM public.tbl_students'
    cursor.execute(query)
    last_loaded_value = cursor.fetchone()[0]

    cursor.close()
    connection.close()

    # if no data has been loaded before, return None
    return last_loaded_value

# **VII. ETL Operations**

In [23]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# main function to run the ETL process with incremental loading
def run_incremental_etl():
    mysql_config = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD', ''),
    'database': os.getenv('MYSQL_DATABASE')
    }

    # New dynamic PostgreSQL configuration
    postgres_config = {
        'host': os.getenv('POSTGRES_HOST', 'localhost'),
        'user': os.getenv('POSTGRES_USER'),
        'password': os.getenv('POSTGRES_PASSWORD', ''),
        'database': os.getenv('POSTGRES_DATABASE')
    }

    # specify the table and column used for incremental load
    updated_at_column = 'updated_at'
    table_name = 'public.tbl_students'

    # step 1: Get the last loaded value (timestamp) from PostgreSQL
    last_loaded_value = get_last_loaded_value(postgres_config, table_name, updated_at_column)

    # if no value is found, set a default value
    if not last_loaded_value:
        last_loaded_value = '1970-01-01 00:00:00'

    # step 2: extract only new or updated data from MySQL
    query = f'''
    SELECT * FROM tbl_students
    WHERE {updated_at_column} > '{last_loaded_value}'
    '''
    df = extract_incremental_from_mysql(mysql_config, query, last_loaded_value)

    # check if there are new or updated records
    if not df.empty:
        # step 3: transform the data
        df_transformed = transform_data(df)

        # step 4: load the new or updated data into PostgreSQL
        load_to_postgres(postgres_config, df_transformed, table_name)
        print(f"Loaded {len(df_transformed)} new or updated records into PostgreSQL.")
    else:
        print("No new or updated records to load.")

    print("Incremental ETL process completed.")

# run the incremental ETL process
if __name__ == '__main__':
    run_incremental_etl()

No new or updated records to load.
Incremental ETL process completed.


# Excellent! You have just completed this Notebook.

This notebook was created by [Chanthan DINA](https://www.linkedin.com/in/chanthandina/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!

## <h3 align="center"> &#169; CamboAI. All rights reserved. <h3/>